# 15 — Timeframes

## Goal
Understand the **three-timeframe hierarchy** and how each layer feeds the next.

---

## The Three Layers

| Layer | Typical TF | Role |
|-------|-----------|------|
| **HTF** | 4H / Daily | Bias and curve position — *where* to look |
| **MTF** | 1H / 15m  | Zone detection — *what* to trade |
| **LTF** | 5m / 1m   | Entry trigger — *when* to enter |

The golden rule: **never trade LTF against HTF bias**.

---

## Resampling

Given a base dataset at 1-min or tick resolution, higher timeframes are built by resampling:

$$\text{HTF\_open}  = \text{first}(\text{open in period})$$
$$\text{HTF\_high}  = \max(\text{high in period})$$
$$\text{HTF\_low}   = \min(\text{low in period})$$
$$\text{HTF\_close} = \text{last}(\text{close in period})$$

In pandas: `df.resample("4H").agg({...})`.

---

## Fixture Note
The fixture dataset is a synthetic intraday series — it does not have real timestamps.  
We simulate the three-layer concept by **partitioning** the 100-candle dataset:

- HTF = the full dataset (macro view)
- MTF = the same dataset (zone detection, already done in NB06–07)
- LTF = individual candles inside each base (micro trigger)

---

## Visualization
Three stacked panels: HTF range lines, MTF zones, and LTF candles side-by-side for Scenario A.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. Simulate timeframe layers on the fixture

In [ ]:
# Simulate HTF: aggregate every 10 candles
RESAMPLE_N = 10
htf_rows = []
for start in range(0, len(df) - RESAMPLE_N + 1, RESAMPLE_N):
    chunk = df.iloc[start:start + RESAMPLE_N]
    htf_rows.append({
        "open":  chunk["open"].iloc[0],
        "high":  chunk["high"].max(),
        "low":   chunk["low"].min(),
        "close": chunk["close"].iloc[-1],
        "atr":   chunk["atr"].mean(),
        "label": df.index[start],
    })
htf_df = pd.DataFrame(htf_rows).set_index("label")

# MTF = original df (zones already detected above)
print(f"HTF bars : {len(htf_df)}")
print(f"MTF bars : {len(df)}")
print(f"Zones    : {len(zones)}")


## 3. Visualization — three-layer chart

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.35, 0.65],
                    subplot_titles=["HTF (10-bar aggregation)", "MTF — zones"],
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(
    x=htf_df.index, open=htf_df["open"], high=htf_df["high"],
    low=htf_df["low"], close=htf_df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="HTF",
), row=1, col=1)

fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR,
    name="MTF", showlegend=True,
), row=2, col=1)

for z in zones:
    x0, x1 = df.index[z["bs"]], df.index[z["be"]]
    c_edge  = EDGE[z["zone_type"]]
    fig.add_vrect(x0=x0, x1=x1, fillcolor="rgba(255,255,255,0.06)",
                  line=dict(color=c_edge, width=1, dash="dot"), row=2, col=1)

fig.update_layout(
    title="Timeframe Hierarchy — HTF / MTF",
    height=600, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
)
for ax in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
    fig.update_layout(**{ax: dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)})
fig.show()
